# Requirements

In [3]:
import os, sys, platform
node = platform.node()
if "arctrd" in node:
    ROOT_DIR = os.getcwd()
else:
    ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(ROOT_DIR)

from omegaconf import OmegaConf, DictConfig
import numpy as np
import pandas as pd
import torch 
import os
import time

import importlib
import assets.scripts.plot_losses 

from src.settings import LOGS_ROOT, DATA_ROOT

from src.datasets.ukb_hold import load_data_hold as load_ukb_pretrain

UKB_DATADICT, demo_df = load_ukb_pretrain(
    file_path=os.path.join(DATA_ROOT, "ukb_ica/ukb_data_hold.npz"),
    demo_path=os.path.join(DATA_ROOT, "ukb_ica/demographics_legend_hold.csv"),
)

UKB_DATA = UKB_DATADICT['data']
print("Available data in ukb datadict:",UKB_DATADICT.keys())
# prepare train and validation sets
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(UKB_DATA, test_size=0.2, random_state=42)

from src.datasets.fbirn import load_data_hold as load_fbirn_hold

FBIRN_DATADICT, fbirn_demo_df = load_fbirn_hold()

FBIRN_DATA = FBIRN_DATADICT['data']
print("Available data in fbirn datadict:",FBIRN_DATADICT.keys())
print("FBIRN data shape:", FBIRN_DATA.shape)

Available data in ukb datadict: dict_keys(['data', 'sexes', 'ages', 'age_bins'])
Available data in fbirn datadict: dict_keys(['data', 'diags', 'sexes', 'ages', 'age_bins'])
FBIRN data shape: (16, 140, 53)


In [6]:
from src.models.DECIFRA import default_HPs
from src.models.DECIFRA import DECIFRA
# from src.models.DECIFRA_I import DECIFRA_I

def derive_matrices_for_checkpoint(ckpt_path):
    cfg = {
        "data_info": {
            "feature_size": UKB_DATA.shape[2],
            "n_classes": 2,
            }
        }
    cfg = OmegaConf.create(cfg)
    model_cfg = default_HPs(cfg)

    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

    model = DECIFRA(model_cfg)
    fbirn_input = torch.tensor(FBIRN_DATA, dtype=torch.float32).to(device)

    ckpt = torch.load(ckpt_path, map_location='cpu')
    state_dict = ckpt.get('state_dict', ckpt)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()

    _, output = model(fbirn_input)

    matrices = output["matrices"].detach().cpu().numpy()
    return matrices

def derive_matrices_for_model(model_cfg, ckpt_path):
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

    # model = DECIFRA_I(model_cfg)
    model = DECIFRA(model_cfg)
    fbirn_input = torch.tensor(FBIRN_DATA, dtype=torch.float32).to(device)

    ckpt = torch.load(ckpt_path, map_location='cpu')
    state_dict = ckpt.get('state_dict', ckpt)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()

    _, output = model(fbirn_input)

    matrices = output["matrices"].detach().cpu().numpy()
    return matrices

import matplotlib.pyplot as plt
import numpy as np
def plot_combined_matrices(matrices, save_path, n_samples=5, n_time=5):
    if n_samples == -1:
         n_samples = matrices.shape[0]
         
    # Normalize the range for the seismic colormap to center around 0
    abs_max = np.max(np.abs(matrices[:n_samples, 50:(n_time+50)]))
    vmin, vmax = -abs_max, abs_max  # Centering colormap around 0

    # Determine the size of individual matrices
    matrix_size = matrices[0, 0].shape[0]

    # Create a large matrix to hold all the smaller matrices with padding
    combined_matrix = np.full(
        ((matrix_size + 1) * n_samples - 1, (matrix_size + 1) * n_time - 1),
        0.1
    )

    for i in range(n_samples):
        for j in range(n_time):
            matrix = matrices[i, 50 + j]  # Convert to NumPy
            start_row = i * (matrix_size + 1)
            start_col = j * (matrix_size + 1)
            combined_matrix[start_row:start_row + matrix_size, start_col:start_col + matrix_size] = matrix

    # Plot the combined matrix
    # dpi=1
    dpi=4
    figsize = combined_matrix.shape[1] * dpi, combined_matrix.shape[0] * dpi  # Match the figure size to the array dimensions (pixels)
    fig = plt.figure(figsize=figsize, dpi=dpi)
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)
    ax.imshow(combined_matrix, cmap="seismic", vmin=vmin, vmax=vmax, interpolation='none')
    # check if save directory exists, if not create it
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=dpi)
    plt.close()

def plot_mean_matrices(matrices, save_path, n_samples=5):
    ### also plot mean-over-time matrices ###
    if n_samples == -1:
         n_samples = matrices.shape[0]
    # Calculate the mean over time for each sample
    mean_matrices = matrices[:n_samples].mean(dim=1)

    # Normalize the range for the seismic colormap to center around 0
    abs_max = mean_matrices.abs().max().item()
    vmin, vmax = -abs_max, abs_max  # Centering colormap around 0

    matrix_size = matrices[0, 0].shape[0]
    combined_matrix = np.full(
        ((matrix_size + 1) * n_samples - 1, matrix_size),
        0.1
    )
    for i in range(n_samples):
            matrix = mean_matrices[i].cpu().detach().numpy()  # Convert to NumPy
            start_row = i * (matrix_size + 1)
            combined_matrix[start_row:start_row + matrix_size, :] = matrix
            
    # Plot the combined matrix
    dpi=4
    figsize = combined_matrix.shape[1] * dpi, combined_matrix.shape[0] * dpi  # Match the figure size to the array dimensions (pixels)
    fig = plt.figure(figsize=figsize, dpi=dpi)
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)
    ax.imshow(combined_matrix, cmap="seismic", vmin=vmin, vmax=vmax, interpolation='none')
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=dpi)
    plt.close()

In [ ]:
# older example
exp_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA_MS-default"

fbirn_outputs = {}
for run in os.listdir(exp_path):
    run_path = os.path.join(exp_path, run)

    # check if train_logs.csv exists
    if not os.path.isfile(os.path.join(run_path, "train_logs.csv")):
        continue

    model_cfg_path = os.path.join(run_path, "model_config.yaml")
    model_cfg = OmegaConf.load(model_cfg_path)

    # get best model index from .txt file
    best_model_idx = pd.read_csv(os.path.join(run_path, "best_epoch.txt"), header=None).iloc[0, 0]

    ckpt_path = os.path.join(run_path, f"checkpoints/model_{best_model_idx}.pt")

    matrices = derive_matrices_for_model(model_cfg, ckpt_path)

    np.save(os.path.join(run_path, "fbirn_matrices.npy"), matrices)
    fbirn_outputs[run] = matrices


for key, matrix in fbirn_outputs.items():
    save_path = os.path.join(exp_path, f"_images/{key}.png")
    plot_combined_matrices(matrix, save_path, n_samples=1)

In [9]:
importlib.reload(assets.scripts.plot_matrices)
from assets.scripts.plot_matrices import get_model_class, load_dataset, derive_matrices_for_model, plot_combined_matrices, plot_mean_matrices, plot_aggregated_matrices
test_data = load_dataset("fbirn")
exp_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA"
n_samples = 1


if not os.path.exists(exp_path):
    print(f"Error: Experiment path {exp_path} does not exist.")

model_outputs = {}
for run in os.listdir(exp_path):
    run_path = os.path.join(exp_path, run)
    if not os.path.isdir(run_path):
        continue

    # check if train_logs.csv exists as a sign of a completed/working run
    if not os.path.isfile(os.path.join(run_path, "train_logs.csv")):
        continue

    model_cfg_path = os.path.join(run_path, "model_config.yaml")
    if not os.path.exists(model_cfg_path):
        continue
        
    model_cfg = OmegaConf.load(model_cfg_path)

    # get best model index from .txt file
    best_epoch_file = os.path.join(run_path, "best_epoch.txt")
    if not os.path.exists(best_epoch_file):
        print(f"No best_epoch.txt found in {run_path}")
        continue
        
    best_model_idx = pd.read_csv(best_epoch_file, header=None).iloc[0, 0]
    ckpt_path = os.path.join(run_path, f"checkpoints/model_{best_model_idx}.pt")
    
    if not os.path.exists(ckpt_path):
        print(f"Checkpoint not found: {ckpt_path}")
        continue

    print(f"Processing run {run}...")
    try:
        matrices = derive_matrices_for_model(model_cfg, ckpt_path, test_data)
        if matrices is not None:
            np.save(os.path.join(run_path, "model_matrices.npy"), matrices)
            model_outputs[run] = matrices
        else:
            print(f"No matrices returned for run {run}.")
    except Exception as e:
        print(f"Error processing run {run}: {e}")

for key, matrix in model_outputs.items():
    save_path = os.path.join(exp_path, f"_images/{key}.png")
    plot_combined_matrices(matrix, save_path, n_samples=n_samples)
    
    mean_save_path = os.path.join(exp_path, f"_images/{key}_mean.png")
    plot_mean_matrices(matrix, mean_save_path, n_samples=n_samples)
    if matrices_list:
        aggregated_save_path = os.path.join(exp_path, f"_images/aggregated_matrices.png")
        plot_aggregated_matrices(matrices_list, titles_list, aggregated_save_path, n_samples=args.samples)
        print(f"Aggregated plot saved to {aggregated_save_path}")
        

Processing run 03...
No best_epoch.txt found in /Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA/04
Processing run 02...
Processing run 00...
Processing run 01...


In [6]:
import importlib
import assets.scripts.plot_losses
from assets.scripts.plot_matrices import get_model_class, get_model_results, \
    load_dataset, derive_matrices_for_model, plot_combined_matrices, \
        plot_mean_matrices, plot_aggregated_matrices
importlib.reload(assets.scripts.plot_matrices)
from assets.scripts.plot_matrices import get_model_class, get_model_results, \
    load_dataset, derive_matrices_for_model, plot_combined_matrices, \
        plot_mean_matrices, plot_aggregated_matrices
        
def inspect_model(exp_path, test_data, use_cache=True, n_samples=5):

    if not os.path.exists(exp_path):
        print(f"Error: Experiment path {exp_path} does not exist.")

    matrices_list = []
    titles_list = []
    losses = []


    for run in sorted(os.listdir(exp_path)):
        run_path = os.path.join(exp_path, run)
        if not os.path.isdir(run_path):
            continue

        print(f"Processing run {run}...")
    
        try:
            matrices, loss, best_idx = get_model_results(run_path, test_data, use_cache=use_cache, n_samples=n_samples)
            if matrices is not None:
                title = f"Run {run} | Best Epoch: {best_idx}"
                if loss is not None:
                    title += f" | Loss: {loss:.4f}"
                    losses.append(loss)
                else:
                    # Append infinity so runs without a valid loss are sorted to the very end
                    losses.append(float('inf')) 
                    
                matrices_list.append(matrices)
                titles_list.append(title)
                
                # Plot individually
                save_path = os.path.join(exp_path, f"_images/{run}.png")
                plot_combined_matrices(matrices, save_path, n_samples=n_samples)
                
                mean_save_path = os.path.join(exp_path, f"_images/{run}_mean.png")
                plot_mean_matrices(matrices, mean_save_path, n_samples=n_samples)
            else:
                print(f"No valid results derived for run {run}.")
        except Exception as e:
            print(f"Error processing run {run}: {e}")

    # --- ADD THIS SORTING LOGIC HERE ---
    if matrices_list and losses:
        # Zip the lists together, sort by the first element (loss), and unzip
        sorted_combined = sorted(zip(losses, matrices_list, titles_list), key=lambda x: x[0])
        losses, matrices_list, titles_list = zip(*sorted_combined)
        
        # zip(*...) returns tuples, so convert them back to lists
        matrices_list = list(matrices_list)
        titles_list = list(titles_list)
    # -----------------------------------

    if matrices_list:
        aggregated_save_path = os.path.join(exp_path, f"_images/aggregated_matrices.png")
        plot_aggregated_matrices(matrices_list, titles_list, aggregated_save_path, n_samples=n_samples)
        print(f"Aggregated plot saved to {aggregated_save_path}")


In [25]:
# vanilla
test_data = load_dataset("fbirn")
exp_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA"
use_cache=True
n_samples = 1

inspect_model(exp_path, test_data, use_cache=use_cache, n_samples=n_samples)


Processing run 00...
Processing run 01...
Processing run 02...
Processing run 03...
Processing run 04...
No valid results derived for run 04.
Processing run _images...
No valid results derived for run _images.
Aggregated plot saved to /Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA/_images/aggregated_matrices.png


In [42]:
# default
test_data = load_dataset("fbirn")
exp_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA_MS-default"
use_cache=True
n_samples = 1

inspect_model(exp_path, test_data, use_cache=use_cache, n_samples=n_samples)


Processing run 00...
Processing run 01...
Processing run 02...
Processing run 03...
Processing run 04...
Processing run 05...
Processing run 06...
Processing run 07...
Processing run 08...
Processing run 09...
Processing run 10...
No valid results derived for run 10.
Processing run 11...
No valid results derived for run 11.
Processing run _images...
No valid results derived for run _images.
Aggregated plot saved to /Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA_MS-default/_images/aggregated_matrices.png


In [43]:
# no-mix
test_data = load_dataset("fbirn")
exp_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA_MS-noMixed"
use_cache=True
n_samples = 1

inspect_model(exp_path, test_data, use_cache=use_cache, n_samples=n_samples)


Processing run 00...
Processing run 01...
Processing run 02...
Processing run 03...
Processing run 04...
Processing run 05...
Processing run 06...
Processing run 07...
Processing run 08...
Processing run 09...
Processing run 10...
No valid results derived for run 10.
Processing run 11...
No valid results derived for run 11.
Processing run _images...
No valid results derived for run _images.
Aggregated plot saved to /Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA_MS-noMixed/_images/aggregated_matrices.png


In [7]:
# inspect separate checkpoints

test_data = load_dataset("fbirn")
exp_path = "/Users/ppopov1/DECIFRA/assets/logs/1_pretrain-ukb-DECIFRA_MS-default/02/"
use_cache=False
n_samples = 1

matrices, losses, best_idxs, titles = [], [], [], []
epochs = [0, 30, 60, 90, 110, 130, 200, 400, 499]
for epoch in epochs:
    matrix, loss, best_idx = get_model_results(exp_path, test_data, checkpoint=epoch, use_cache=False, n_samples=n_samples)
    matrices.append(matrix)
    losses.append(loss)
    best_idxs.append(best_idx)

    title = f"Epoch {epoch:04d} | Loss: {loss:.4f}"
    titles.append(title)
    


In [8]:

aggregated_save_path = "/Users/ppopov1/DECIFRA/scripts/assets/plots/multistage_inscpect.png"
plot_aggregated_matrices(matrices, titles, aggregated_save_path, n_samples=n_samples)
print(f"Aggregated plot saved to {aggregated_save_path}")

Aggregated plot saved to /Users/ppopov1/DECIFRA/scripts/assets/plots/multistage_inscpect.png
